In [ ]:
%reload_ext autoreload
%autoreload 2

In [ ]:
import numpy as np
from sklearn.datasets import make_classification
from sklearn.linear_model import LogisticRegression
from mapie.risk_control import MultiClassificationController
from mapie.risk_control.methods import ltt_procedure
from sklearn.metrics import average_precision_score
import plotly.express as px
import plotly.graph_objects as go

X_toy, y_toy = make_classification(
            n_samples=300,
            n_features=5,
            n_informative=5,
            n_redundant=0,
            n_repeated=0,
            n_classes=5,
            n_clusters_per_class=1,
            #weights=[0.5, 0.5],
            flip_y=0,
            random_state=42
        )
#X_toy, y_toy = make_gaussian_quantiles(
#            n_samples=1000,
#            n_features=2,
#           n_classes=5,
#            #weights=[0.5, 0.5],
#        )

In [ ]:
lr = LogisticRegression()
clf = LogisticRegression().fit(X_toy, y_toy)

In [ ]:
y_train_pred = clf.predict_proba(X_toy)
print(average_precision_score(y_toy,y_train_pred))

In [ ]:
# regarder avec la valeur des p values. 

In [ ]:
def funct_risk_combination_selector(input):
    output = input[1,:]
    return output 

In [ ]:
np.linspace(0.9, 0.05, 10)

In [ ]:
from mapie.risk_control.risks import build_precision_ovr

In [ ]:
build_precision_ovr(np.unique(y_toy))

In [ ]:
mapie_clf = MultiClassificationController(predict_function=clf.predict_proba, target_level=0.75,risks=build_precision_ovr(np.unique(y_toy)),
                                          confidence_level = 0.9,list_predict_params=np.linspace(0.05, 0.9, 90),
                                          fwer_method="split_fixed_sequence",
                                          risk_combination_method=1)#
#mapie_clf.calibrate(X_toy, y_toy)

In [ ]:
y_calibrate = y_toy
X_calibrate = X_toy

In [ ]:
mapie_clf.learn_fixed_sequence_order(X_calibrate,y_calibrate)

In [ ]:
mapie_clf.calibrate(X_toy, y_toy)

In [ ]:
y_calibrate_ = np.asarray(y_calibrate, dtype=int)
mapie_clf.risk_combination_method = mapie_clf._check_risk_combination_method(
    mapie_clf.risk_combination_method, y_calibrate
)
original_params = mapie_clf._predict_params
if mapie_clf.fwer_method == "split_fixed_sequence":
    if mapie_clf._learned_fixed_sequence is None:
        raise ValueError(
            "You must call 'learn_fixed_sequence_order' before 'calibrate' "
            "when using fwer_method='split_fixed_sequence'."
        )
    mapie_clf._predict_params = mapie_clf._learned_fixed_sequence

predictions_per_param = mapie_clf._get_predictions_per_param(
    X_calibrate, mapie_clf._predict_params, is_calibration_step=True
)
# populate multirisks
class_labels = np.unique(y_calibrate)
#[risk.populate_risk_per_class(class_labels) for risk in mapie_clf._risk]
# broadcast one _alpha_level per class.


In [ ]:
predictions_per_param

In [ ]:
# non classe => nan car classe 0 existe déjà. 
predictions_per_param.shape

In [ ]:
risk_values, eff_sample_sizes = mapie_clf._get_risk_values_and_eff_sample_sizes(
    y_calibrate_, predictions_per_param, mapie_clf._risk
)

In [ ]:
(valid_index, p_values) = ltt_procedure(
            risk_values,
            np.expand_dims(mapie_clf._alpha, axis=1),
            mapie_clf._delta,
            eff_sample_sizes,
            binary = True,
            fwer_method=mapie_clf.fwer_method,
        )

In [ ]:
p_values

In [ ]:

fig = px.line(p_values, title='plot of an array  : y=p_values , x = index')

n_lambdas = p_values.shape[0]
for j, list_index in enumerate(valid_index):
    colors = []
    for index in range(n_lambdas):
        if index in list_index:
            colors.append("green")
        else:
            colors.append("red")
    fig.add_trace(go.Scatter(
        x=list(range(n_lambdas)),
        y=p_values[:, j],
        mode="markers",
        marker=dict(color=colors),
        showlegend=False,
    ))

fig.show()